# Session 4 · Exploring YARN and Classic MapReduce

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Diploma-in-Data-Science/blob/main/term-2/02-big-data-systems-with-hadoop/code/session-04-yarn-and-classic-mapreduce.ipynb)

Diploma in Data Science · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run the cells from top to bottom. Every example is self-contained, so you can also run any single cell on its own.

### Example 1 · Slots vs containers

In [1]:
node_memory_gb = 12
phases = [("map-heavy phase", 10, 0), ("mixed phase", 4, 4), ("reduce-heavy phase", 0, 8)]

print(f"{'phase':<20}{'MRv1 busy slots':>16}{'YARN busy containers':>22}")
for name, maps, reduces in phases:
    mrv1 = min(maps, 4) + min(reduces, 2)           # fixed 4 map + 2 reduce slots
    yarn = min(maps + reduces, node_memory_gb // 2)  # any task fits a 2 GB container
    print(f"{name:<20}{mrv1:>10} of 6{yarn:>16} of 6")

phase                MRv1 busy slots  YARN busy containers
map-heavy phase              4 of 6               6 of 6
mixed phase                  6 of 6               6 of 6
reduce-heavy phase           2 of 6               6 of 6


### Example 2 · FIFO vs Fair scheduling

In [2]:
def simulate(policy, capacity=10):
    jobs = {"big_batch": {"arrive": 0, "left": 100, "done": None},
            "small_query": {"arrive": 1, "left": 5, "done": None}}
    t = 0
    while any(j["left"] for j in jobs.values()):
        active = [n for n, j in jobs.items() if j["arrive"] <= t and j["left"]]
        share = {n: 0 for n in active}
        free = capacity
        if policy == "FIFO":
            for n in sorted(active, key=lambda n: jobs[n]["arrive"]):
                share[n] = min(free, jobs[n]["left"]); free -= share[n]
        else:  # Fair: split equally, give leftovers to whoever still needs them
            while free and any(share[n] < jobs[n]["left"] for n in active):
                for n in active:
                    if free and share[n] < jobs[n]["left"]:
                        share[n] += 1; free -= 1
        t += 1
        for n, s in share.items():
            jobs[n]["left"] -= s
            if jobs[n]["left"] == 0 and jobs[n]["done"] is None:
                jobs[n]["done"] = t
    return {n: j["done"] - j["arrive"] for n, j in jobs.items()}

for policy in ["FIFO", "Fair"]:
    r = simulate(policy)
    print(f"{policy:<5} turnaround  big_batch={r['big_batch']:>3}  small_query={r['small_query']:>3}")

FIFO  turnaround  big_batch= 10  small_query= 10
Fair  turnaround  big_batch= 11  small_query=  1


### Example 3 · Capacity Scheduler queues

In [3]:
cluster_containers = 200
queues = {"etl": (50, 100), "analytics": (30, 100), "adhoc": (20, 40)}  # (capacity %, max %)
demand = {"etl": 40, "analytics": 150, "adhoc": 120}

alloc = {q: min(demand[q], cluster_containers * c // 100) for q, (c, _) in queues.items()}
free = cluster_containers - sum(alloc.values())
print("guaranteed:", alloc, "| idle:", free)
for q, (c, mx) in sorted(queues.items(), key=lambda x: -demand[x[0]]):   # lend idle capacity (elasticity)
    extra = min(free, demand[q] - alloc[q], cluster_containers * mx // 100 - alloc[q])
    if extra > 0:
        alloc[q] += extra; free -= extra
print("after elasticity:", alloc, "| idle:", free)

guaranteed: {'etl': 40, 'analytics': 60, 'adhoc': 40} | idle: 60
after elasticity: {'etl': 40, 'analytics': 120, 'adhoc': 40} | idle: 0
